# Reasoning Memory — Remember WHY You Decided, Not Just What You Know

The earlier demos store what the agent knows. This demo stores **why it decided**: the question → tool steps → evidence → outcome chain, with provenance. Without it, "why did you recommend X?" gets a confabulated answer — the agent invents a plausible justification because the real reasoning was never kept.

> **Honesty note:** "reasoning memory" is an **engineering pattern**, not an established category in academic memory taxonomies. What the research does support is the value of traceability and provenance in agent memory — MemWeaver ([arXiv 2601.18204](https://arxiv.org/abs/2601.18204)) and the Engram system ([arXiv 2606.09900](https://arxiv.org/abs/2606.09900), single-author preprint).

Two tracks, same traces:
- **Key-value** (`agent.state`): a Strands `HookProvider` records each invocation's trace automatically — zero changes to the tools.
- **Graph** (Neo4j): the same traces as node chains with provenance edges — enables the **reverse audit** a flat store can't express.

This demo uses Strands Agents. The patterns are framework-agnostic and carry over to other agent frameworks.

## Prerequisites

1. `OPENAI_API_KEY` (used by the model, gpt-4o-mini)
2. For the graph tests: a running Neo4j (Desktop, Docker, or Aura) with `NEO4J_*` values in `.env`

```bash
uv venv && uv pip install -r requirements.txt
cp .env.example .env
```

## Install dependencies

Run this once (or install from a terminal with `uv venv && uv pip install -r requirements.txt`).

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


## Configure your model provider

The demo runs with **OpenAI** by default, but you can use **Amazon Bedrock**, **Anthropic**, or any provider available in the Strands configuration — see [supported model providers](https://strandsagents.com/docs/user-guide/concepts/model-providers/?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el).

- **OpenAI (default):** set `OPENAI_API_KEY` below or in a `.env` file. Get one at https://platform.openai.com/api-keys
- **Amazon Bedrock:** no OpenAI key needed — uses your AWS credentials (`aws configure`, with model access enabled in your region). In the model cell below, comment the `OpenAIModel` lines and uncomment the Bedrock block.

In [2]:
import os

# python-dotenv loads OPENAI_API_KEY (and any other config) from a local .env file,
# so you don't have to export variables in every new terminal or notebook kernel.
from dotenv import load_dotenv
load_dotenv()

# os.environ['OPENAI_API_KEY'] = 'your-key-here'  # Or uncomment and set your key here (not needed for Bedrock)

USING_OPENAI = True  # set False if you switch to Bedrock in the model cell below
if USING_OPENAI:
    assert os.getenv('OPENAI_API_KEY'), (
        'OPENAI_API_KEY not set. '
        'Get yours at https://platform.openai.com/api-keys and set it above or in a .env file — '
        'or switch to Amazon Bedrock in the model cell below.'
    )
print('Provider configured')

Provider configured


## Create the model

Just the engine — one model object, reused by every agent in this notebook.

In [3]:
# OTEL_SDK_DISABLED silences OpenTelemetry tracing noise in notebook output.
os.environ['OTEL_SDK_DISABLED'] = 'true'

# Using OpenAI-compatible interface via Strands SDK (not direct OpenAI usage)
from strands.models.openai import OpenAIModel

MODEL = OpenAIModel(model_id='gpt-4o-mini')  # api_key read from the OPENAI_API_KEY env var

# To run on Amazon Bedrock instead (no OpenAI key; uses your AWS credentials),
# comment the two lines above and uncomment these two:
# from strands.models import BedrockModel
# MODEL = BedrockModel(model_id='openai.gpt-oss-120b-1:0', region_name='us-west-2')

print('Model ready')

Model ready


---
## Test 1 — The problem: the reasoning is lost

The agent decides with tools. A **later session** — which only kept the outcome, like demos 01-03 would — is asked WHY. It has no trace to consult, so whatever it answers is reconstruction, not the real chain.

In [4]:
# Agent is the Strands agent loop: it calls the model, runs tools, and loops until done.
from strands import Agent

# trace_kv holds this demo's standalone track: the deterministic demo tools
# (search_flights, check_fare_alert) and, later, the trace recorder.
import trace_kv as kv

# The assistant whose decisions we want to be able to explain later.
SYSTEM_PROMPT = 'You are a travel assistant. Use your tools to answer. Be concise — 2-3 sentences maximum.'

deciding_agent = Agent(model=MODEL, system_prompt=SYSTEM_PROMPT,
                       tools=[kv.search_flights, kv.check_fare_alert], callback_handler=None)
decision = deciding_agent("Find me a flight JFK to Madrid on 2026-10-10 and pick the best option.")
print("Decision made:", str(decision).strip())

Decision made: The best flight option from JFK to Madrid on 2026-10-10 is with American Airlines for $223.25 USD, departing at 06:58 AM and arriving at 09:13 PM (direct, 8h 15m duration). There are currently no fare alerts for this route.


In [5]:
# A later session: only the outcome survived. The reasoning chain was never persisted.
later_agent = Agent(model=MODEL, system_prompt=SYSTEM_PROMPT, callback_handler=None)
answer = later_agent("Earlier you told me: 'I recommend the Iberia JFK-MAD flight.' Why did you recommend that Madrid flight?")
print("Asked WHY:", str(answer).strip())

steps_recoverable = len(later_agent.state.get(kv.TRACES_KEY) or [])
print("\nReal reasoning steps recoverable from the store:", steps_recoverable)
print("Whatever the answer says, it is a plausible reconstruction — not the real chain.")

Asked WHY: I recommended the Iberia JFK-MAD flight due to its convenience, frequent schedule, and competitive pricing. Iberia is known for its quality service, which can enhance your overall travel experience to Madrid.

Real reasoning steps recoverable from the store: 0
Whatever the answer says, it is a plausible reconstruction — not the real chain.


---
## Test 2 — The recorder: a HookProvider captures the trace automatically

Same tools, same question. The only change is `Agent(hooks=[DecisionTraceRecorder()])`. The recorder subscribes to the lifecycle events every tool call already emits — **zero changes to the tools** — and assembles one trace per invocation into `agent.state`.

In [6]:
# DecisionTraceRecorder is a Strands HookProvider: it subscribes to the lifecycle
# events every tool call already emits, so it records traces with ZERO tool changes.
# why_did_i is the replay tool — it reads the recorded trace back from agent.state.
agent = Agent(model=MODEL, system_prompt=SYSTEM_PROMPT,
              tools=[kv.search_flights, kv.check_fare_alert, kv.why_did_i],
              hooks=[kv.DecisionTraceRecorder()], callback_handler=None)
decision = agent("Find me a flight JFK to Madrid on 2026-10-10 and pick the best option.")
print("Decision made:", str(decision).strip())

traces = agent.state.get(kv.TRACES_KEY) or []
print(f"\nRecorded automatically: {len(traces)} trace(s), {sum(len(t['steps']) for t in traces)} step(s)")
for step in traces[0]["steps"]:
    print(f"  step {step['n']}: {step['tool']}({step['input']}) -> {step['evidence']['text']}")

Decision made: The best flight option from JFK to Madrid on 2026-10-10 is with American Airlines for $222.22 (USD) in economy class. It departs at 06:58 AM and arrives at 09:13 PM, with a flight duration of 8 hours and 15 minutes, and no stops.

Recorded automatically: 1 trace(s), 2 step(s)
  step 1: search_flights({'origin': 'JFK', 'destination': 'MAD', 'departure_date': '2026-10-10'}) -> [
 {
  "offer_id": "off_0000B8RT1nck9D0Sijw5cz",
  "price": 222.22,
  "currency": "USD",
  "cabin": "economy",
  "slices": [
   {
    "origin": "JFK",
    "destination": "MAD",
    "segments": [
     {
      "carrier": "American Airlines",
      "flight_number": "4",
      "departing_at": "2026-10-10T06:58:00",
      "arriving_at": "2026-10-10T21:13:00",
      "duration": "PT8H15M"
     }
    ],
    "stops": 0
   }
  ]
 },
 {
  "offer_id": "off_0000B8RT1nbgDA9ifRREyA",
  "price": 222.62,
  "currency": "USD",
  "cabin": "economy",
  "slices": [
   {
    "origin": "JFK",
    "destination": "MAD",
    "

In [7]:
# Now the agent can replay its own REAL chain via the why_did_i tool.
answer = agent("Why did you recommend that Madrid flight?")
print("Asked WHY (agent replays its own trace):", str(answer).strip())

trace = kv.replay_why(agent.state.get(kv.TRACES_KEY) or [], "Madrid")
print(f"\nReal reasoning steps recoverable from the store: {len(trace['steps'])}")

Asked WHY (agent replays its own trace): I recommended the American Airlines flight from JFK to Madrid because it was the cheapest option available at $222.22 (USD) in economy class, with a direct route and a convenient flight duration of 8 hours and 15 minutes. Additionally, there were no fare alerts indicating it was an unusually good deal.

Real reasoning steps recoverable from the store: 2


---
## Test 3 — The graph track: traces as node chains with provenance edges

The same traces stored in Neo4j:

```
(:Decision)-[:HAS_STEP]->(:Step)-[:NEXT]->(:Step)     the reasoning chain
(:Step)-[:USED]->(:Evidence)                           what each step relied on
(:Evidence)-[:DERIVED_FROM]->(:Evidence)               evidence built on other evidence
(:Evidence)-[:FROM_SOURCE]->(:Source)                  external origin
```

The demo seeds a known 5-decision history (deterministic, reproducible): two flight choices, a budget built **on those choices' outputs**, an itinerary built **on the budget**, and a weather-based packing list as the control.

In [8]:
# trace_graph is the Neo4j track: the same traces stored as node chains with
# provenance edges, in this demo's own isolated database (reasoningdemo).
import trace_graph as tg

driver = tg.get_driver()
db = tg.ensure_database(driver)
tg.reset_graph(driver, db)
summary = tg.seed_graph(driver, db)
print(f"Seeded: {summary['decisions']} decisions, {summary['evidence']} evidence records, {summary['sources']} external sources")

Seeded: 5 decisions, 8 evidence records, 3 external sources


In [9]:
# Replay works the same as the flat store — one traversal instead of one lookup.
trace = tg.replay_why_graph(driver, db, "Iberia")
print("Replay 'why the Madrid flight?':", trace["question"])
for step in trace["steps"]:
    print(f"  step {step['n']}: {step['tool']}({step['input']}) -> {step['evidence']}")
print("Outcome:", trace["outcome"])

Replay 'why the Madrid flight?': Which flight should I book to Madrid?
  step 1: search_flights({'origin': 'JFK', 'destination': 'MAD'}) -> Candidates JFK-MAD: Iberia non-stop, TAP one-stop via Lisbon.
  step 2: check_fare_alert({'route': 'JFK-MAD'}) -> Fare alert: Iberia JFK-MAD at $612 is 30% below typical.
Outcome: Recommended the Iberia non-stop JFK-MAD.


---
## Test 4 — The reverse audit: "this source was wrong — which decisions relied on it?"

The fare-alerts feed is declared compromised. Ground truth by construction: **4 of 5** decisions depend on it — 2 directly, 2 only through other decisions' outputs.

- The **flat scan** checks each trace's own blob → finds only direct citations.
- The **graph traversal** follows `DERIVED_FROM*0..` → finds them all, at any depth, with the evidence path as a receipt.

In [10]:
ground_truth = sorted(kv.AFFECTED_IDS)
print(f"Ground truth: {len(ground_truth)} affected decisions {ground_truth}\n")

kv_found = sorted(kv.find_affected_decisions_kv(kv.SEED_TRACES, kv.COMPROMISED_SOURCE))
print(f"Flat scan (key-value): found {len(kv_found)}/{len(ground_truth)}  {kv_found}")
missed = sorted(set(ground_truth) - set(kv_found))
print(f"Missed (indirect deps): {missed}\n")

graph_found = sorted(tg.find_affected_decisions_graph(driver, db))
print(f"Graph traversal: found {len(graph_found)}/{len(ground_truth)}  {graph_found}")
print(f"Control '{kv.CONTROL_ID}' correctly NOT flagged by either:",
      kv.CONTROL_ID not in kv_found and kv.CONTROL_ID not in graph_found)

Ground truth: 4 affected decisions ['madrid-flight', 'tokyo-flight', 'trip-budget', 'trip-itinerary']

Flat scan (key-value): found 2/4  ['madrid-flight', 'tokyo-flight']
Missed (indirect deps): ['trip-budget', 'trip-itinerary']

Graph traversal: found 4/4  ['madrid-flight', 'tokyo-flight', 'trip-budget', 'trip-itinerary']
Control 'packing-list' correctly NOT flagged by either: True


In [11]:
# The receipts — the exact evidence path from each indirect decision to the source.
for decision_id in missed:
    hops = tg.provenance_path(driver, db, decision_id)
    print(f"{decision_id}: {' -> '.join(hops)}")

driver.close()

trip-budget: trip-budget -> trip-budget-step-2 -> tokyo-fare -> tokyo-fare-alert -> fare_alerts_feed
trip-itinerary: trip-itinerary -> trip-itinerary-step-1 -> itinerary-draft -> madrid-fare -> madrid-fare-alert -> fare_alerts_feed


---
## Summary

| Store | Replay "why did I decide X?" | Reverse audit "source S was wrong" |
|-------|------------------------------|-------------------------------------|
| **Key-value** (flat scan) | ✅ one lookup | **2/4** — direct citations only |
| **Graph** (traversal) | ✅ one traversal | **4/4** — any depth, with receipts |

**Key insight:** both stores replay individual decisions equally well. The graph earns its keep on the *reverse* question — a flat blob never mentions sources it depends on indirectly, while the provenance traversal follows evidence through other decisions' outputs.

![Reverse audit](images/reasoning-memory-reverse-audit.png)

**Scope notes:** the recorder captures tool calls and outcomes, not the model's internal chain-of-thought. "Reasoning memory" is an engineering pattern; the research-backed theme is traceability/provenance (MemWeaver, Engram). All numbers are deterministic checks against the seeded history — no LLM judge.